# 01 — Colab setup and Olist data ingestion

Prepare a reproducible Colab environment, download the Olist Brazilian E-Commerce dataset, validate its nine source files, and persist the original CSVs in Google Drive.

In [ ]:
%pip -q install kagglehub pandas pyarrow pyspark

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
PROJECT = Path("/content/drive/MyDrive/ecommerce-data-engineering")
RAW = PROJECT / "data" / "raw"
REPORTS = PROJECT / "reports"
for folder in [RAW, REPORTS, PROJECT / "data" / "bronze"]:
    folder.mkdir(parents=True, exist_ok=True)
print(PROJECT)

In [ ]:
import kagglehub
download_dir = Path(kagglehub.dataset_download("olistbr/brazilian-ecommerce"))
csv_paths = list(download_dir.rglob("*.csv"))
print("CSV files discovered:", len(csv_paths))
for p in sorted(csv_paths):
    print(p.name, f"{p.stat().st_size / 1024**2:.2f} MB")

If Kaggle download fails, download and unzip the [Olist dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce), then upload the nine CSV files to `MyDrive/ecommerce-data-engineering/data/raw/`.

In [ ]:
import shutil
EXPECTED_FILES = {
    "olist_customers_dataset.csv", "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv", "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv", "olist_orders_dataset.csv",
    "olist_products_dataset.csv", "olist_sellers_dataset.csv",
    "product_category_name_translation.csv",
}
discovered = {p.name:p for p in download_dir.rglob("*.csv")}
missing = EXPECTED_FILES - set(discovered)
if missing:
    raise FileNotFoundError(f"Missing files: {sorted(missing)}")
for name in sorted(EXPECTED_FILES):
    src, dst = discovered[name], RAW / name
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        shutil.copy2(src, dst)
assert {p.name for p in RAW.glob("*.csv")} == EXPECTED_FILES
print("SUCCESS: 9 source files saved:", RAW)

In [ ]:
import pandas as pd
inventory = []
for path in sorted(RAW.glob("*.csv")):
    df = pd.read_csv(path, low_memory=False)
    inventory.append({
        "file_name": path.name, "row_count": len(df),
        "column_count": len(df.columns),
        "missing_cells": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
        "size_mb": round(path.stat().st_size / 1024**2, 3)
    })
    del df
source_inventory = pd.DataFrame(inventory).sort_values("file_name")
display(source_inventory)
source_inventory.to_csv(REPORTS / "source_inventory.csv", index=False)

In [ ]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder.master("local[2]")
    .appName("olist-ecommerce-data-engineering")
    .config("spark.driver.memory","2g")
    .config("spark.sql.shuffle.partitions","4")
    .config("spark.sql.session.timeZone","UTC").getOrCreate())
orders_spark = (spark.read.option("header",True).option("inferSchema",True)
    .csv(str(RAW / "olist_orders_dataset.csv")))
orders_spark.printSchema()
print("Order records:", orders_spark.count())
orders_spark.show(5, truncate=False)

**Expected:** approximately 99,441 orders. Confirm all nine source files and the inventory report exist in Drive. Run notebook 02 next.